In [1]:
import pandas as pd  
import numpy as np
from tqdm import tqdm  
from collections import defaultdict  
import os, math, warnings, math, pickle
from tqdm import tqdm
import faiss
import collections
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder
from datetime import datetime
from deepctr.feature_column import SparseFeat, VarLenSparseFeat
from sklearn.preprocessing import LabelEncoder
from tensorflow.python.keras import backend as K
from tensorflow.python.keras.models import Model
# from tensorflow.python.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.sequence import pad_sequences
from deepmatch.models import *
from deepmatch.utils import sampledsoftmaxloss
warnings.filterwarnings('ignore')

import gc

from deepmatch.models import YoutubeDNN
import tensorflow as tf
from tensorflow.keras.layers import *


from tensorflow.keras.models import Model

DeepCTR version 0.9.3 detected. Your version is 0.8.2.
Use `pip install -U deepctr` to upgrade.Changelog: https://github.com/shenweichen/DeepCTR/releases/tag/v0.9.3


In [2]:
data_path = './data/'
save_path = './temp_results/'
# 做召回评估的一个标志, 如果不进行评估就是直接使用全量数据进行召回
metric_recall = True

In [3]:
# debug
def get_all_click_sample(data_path, sample_nums = 10000):
    all_click = pd.read_csv(data_path + 'train_click_log.csv')
    all_user_ids = all_click.user_id.unique()
    sample_user_ids = np.random.choice(all_user_ids, size = sample_nums, replace=False)
    all_click = all_click[all_click['user_id'].isin(sample_user_ids)]
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click

def get_all_click_df(data_path = data_path, offline = True):
    if offline:
        all_click = pd.read_csv(data_path + 'train_click_log.csv')
    else:
        trn_click = pd.read_csv(data_path + 'train_click_log.csv')
        tst_click = pd.read_csv(data_path + 'testA_click_log.csv')
        all_click = trn_click.append(tst_click)
    all_click = all_click.drop_duplicates((['user_id', 'click_article_id', 'click_timestamp']))
    return all_click
    
def get_item_info_df(data_path):
    item_info_df = pd.read_csv(data_path + 'articles.csv')
    # 方便与训练集中的click_article_id拼接
    item_info_df = item_info_df.rename(columns={'article_id': 'click_article_id'})
    return item_info_df
def get_item_emb_dict(data_path):
    item_emb_df = pd.read_csv(data_path + 'articles_emb.csv')
    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols])
    # 归一化， 将每个向量变成长度=1
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)
    item_emb_dict = dict(zip(item_emb_df['article_id'], item_emb_np))
    pickle.dump(item_emb_dict, open(save_path + 'item_content_emb.pkl', 'wb'))
    return item_emb_dict

In [4]:
max_min_scaler = lambda x: (x-np.min(x))/(np.max(x) - np.min(x))

# 采样数据集
all_click_df = get_all_click_sample(data_path)
# 全量
# all_click_df = get_all_click_df(offline=False)
all_click_df['click_timestamp'] = all_click_df[['click_timestamp']].apply(max_min_scaler)

item_info_df = get_item_info_df(data_path)
item_emb_dict = get_item_emb_dict(data_path)

DeepMatch version 0.3.1 detected. Your version is 0.2.0.
Use `pip install -U deepmatch` to upgrade.Changelog: https://github.com/shenweichen/DeepMatch/releases/tag/v0.3.1


In [5]:
# 根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
def get_user_item_time(click_df):
    click_df = click_df.sort_values('click_timestamp')
    def make_item_time_pair(df):
        return list(zip(df['click_article_id'], df['click_timestamp']))
    user_item_time_df = click_df.groupby('user_id')[['click_article_id', 'click_timestamp']].apply(
        lambda x: make_item_time_pair(x)
    ).reset_index().rename(columns = {0: 'item_time_list'})
    user_item_time_dict = dict(zip(user_item_time_df['user_id'], user_item_time_df['item_time_list']))
    return user_item_time_dict

# 根据时间获取商品被点击的用户序列 {item1: {user1: time1, user2: time2...}...}
def get_item_user_time_dict(click_df):
    def make_user_time_pair(df):
        return list(zip(df['user_id'], df['click_timestamp']))
    click_df = click_df.sort_values('click_timestamp')
    item_user_time_df = click_df.groupby('click_article_id')[['user_id', 'click_timestamp']].apply(lambda x: make_user_time_pair(x)).reset_index().rename(columns={0: 'user_time_list'})
    item_user_time_dict = dict(zip(item_user_time_df['click_article_id'], item_user_time_df['user_time_list']))
    return item_user_time_dict

In [6]:
# 获取当前数据的历史点击、最后一次点击
def get_hist_and_last_click(all_click):
    all_click = all_click.sort_values(by=['user_id', 'click_timestamp'])
    click_last_df = all_click.groupby('user_id').tail(1)
    # 如果用户只有一个点击，hist为空了，会导致训练的时候这个用户不可见，此时默认泄露一下
    def hist_func(user_df):
        if len(user_df) == 1:
            return user_df
        else:
            return user_df[:-1]
    click_hist_df = all_click.groupby('user_id').apply(hist_func).reset_index(drop=True)
    return click_hist_df, click_last_df

# 获取文章属性
def get_item_info_dict(item_info_df):
    max_min_scaler = lambda x : (x-np.min(x))/(np.max(x)-np.min(x))
    item_info_df['created_at_ts'] = item_info_df[['created_at_ts']].apply(max_min_scaler)
    
    item_type_dict = dict(zip(item_info_df['click_article_id'], item_info_df['category_id']))
    item_words_dict = dict(zip(item_info_df['click_article_id'], item_info_df['words_count']))
    item_created_time_dict = dict(zip(item_info_df['click_article_id'], item_info_df['created_at_ts']))
    
    return item_type_dict, item_words_dict, item_created_time_dict

# 获取用户历史点击的文章信息
def get_user_hist_item_info_dict(all_click):
    # 获取user_id对应的用户历史点击文章类型
    user_hist_item_typs = all_click.groupby('user_id')['category_id'].agg(set).reset_index()
    user_hist_item_typs_dict = dict(zip(user_hist_item_typs['user_id'], user_hist_item_typs['category_id']))
    # 获取user_id对应的用户点击文章集合
    user_hist_item_ids_dict = all_click.groupby('user_id')['click_article_id'].agg(set).reset_index()
    user_hist_item_ids_dict = dict(zip(user_hist_item_ids_dict['user_id'], user_hist_item_ids_dict['click_article_id']))
    # 获取user_id对应的用户历史点击文字的平均数
    user_hist_item_words = all_click.groupby('user_id')['words_count'].agg('mean').reset_index()
    user_hist_item_words_dict = dict(zip(user_hist_item_words['user_id'], user_hist_item_words['words_count']))
    # 获取user_id对应的用户最后一次点击的文章的创建时间
    all_click_ = all_click.sort_values('click_timestamp')
    user_last_item_created_time = all_click_.groupby('user_id')['created_at_ts'].apply(lambda x:x.iloc[-1]).reset_index()

    max_min_scaler = lambda x: (x-np.min(x))/(np.max(x)- np.min(x))
    user_last_item_created_time['created_at_ts'] = user_last_item_created_time[['created_at_ts']].apply(max_min_scaler)
    user_last_item_created_time_dict = dict(zip(user_last_item_created_time['user_id'], user_last_item_created_time['created_at_ts']))
    return user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict

# 近期点击次数最多的top-k文章
def get_item_topk_click(click_df, k):
    topk_click = click_df['click_article_id'].value_counts().index[:k]
    return topk_click


## 多路召回字典

In [7]:
# 文章属性信息
item_type_dict, item_words_dict, item_created_time_dict = get_item_info_dict(item_info_df)
# 多路召回字典，用于保存各路召回结果
user_multi_recall_dict = {
    'item_sim_itemcf_recall':{},
    'embedding_sim_item_recall':{},
    'youtubednn_recall':{},
    'youtubednn_usercf_recall':{},
    'cold_start_recall':{}
}

In [8]:
# 提取最后一次点击作为召回评估，如果不需要做召回评估直接使用全量的训练集进行召回(线下验证模型)
# 如果不是召回评估，直接使用全量数据进行召回，不用将最后一次提取出来
trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)

## 召回效果评估    
对当前的召回方法评估，为参数调整指方向

# 评估召回的前10， 20， 30， 40， 50个文章的击中率
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=10):
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    user_num = len(user_recall_items_dict)
    for k in range(10, topk+1, 10):
        hit_num = 0 #命中次数
        # 遍历每个用户的推荐列表
        for user, item_list in user_recall_items_dict.items():
            #用户前k个推荐文章
            tmp_recall_items = [x[0] for x in user_recall_items_dict[user][:k]]
            if last_click_item_dict[user] in set(tmp_recall_items):
                hit_num += 1
        # 求命中率
        hit_rate = round(hit_num * 1.0 / user_num, 5)
        print('topk:', k, ':', 'hit_num', hit_num, 'hit_rate', hit_rate, 'user_num', user_num)

In [9]:
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=10):
    print("✅ 开始评估召回效果...")  # <-- 必加，确保函数执行了
    
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    user_num = len(user_recall_items_dict)

    # 固定评估 top10, top20... 不管你传什么，都能输出！
    for k in [10, 20, 30, 40, 50]:
        hit_num = 0
        for user, item_list in user_recall_items_dict.items():
            # 取前K个召回文章
            tmp_recall_items = [x[0] for x in item_list[:k]]
            if last_click_item_dict.get(user, -1) in tmp_recall_items:
                hit_num += 1
        
        hit_rate = round(hit_num / user_num, 5) if user_num > 0 else 0
        print(f'✅ Top@{k} | 命中数: {hit_num} | 召回率: {hit_rate} | 用户数: {user_num}')

## 求相似性矩阵

kdd 2020 去偏商品推荐
求相似度矩阵时还考虑到 用户点击的时间权重、用户点击顺序权重、文章创建时间权重

In [10]:
# itemcf
def itemcf_sim(df, item_created_time_dict):
    user_item_time_dict = get_user_item_time(df)
    # 求物品相似度
    i2i_sim = {}
    item_cnt = defaultdict(int)
    for user, item_time_list in tqdm(user_item_time_dict.items()):
        for loc1, (i, i_click_time) in enumerate(item_time_list):   # i为当前文章
            item_cnt[i] += 1   # 每个文章被多少用户点击过
            i2i_sim.setdefault(i, {})
            for loc2, (j, j_click_time) in enumerate(item_time_list): # j为同一用户点击过的另一个文章
                if i==j:
                    continue
                # 文章正向顺序点击和反向顺序点击
                loc_alpha = 1.0 if loc2>loc1 else 0.7
                # 位置信息权重
                loc_weight = loc_alpha * (0.9 ** (np.abs(loc2-loc1)-1))
                # 点击时间权重
                click_time_weight = np.exp(0.7 ** np.abs(i_click_time - j_click_time))
                # 两篇文章创建时间的权重
                created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                i2i_sim[i].setdefault(j,0)
                i2i_sim[i][j] += loc_weight * click_time_weight * created_time_weight / math.log(len(item_time_list) + 1)
    i2i_sim_ = i2i_sim.copy()
    for i, related_items in i2i_sim.items():
        for j, wij in related_items.items():
            i2i_sim_[i][j] = wij/math.sqrt(item_cnt[i] * item_cnt[j])
    pickle.dump(i2i_sim_, open(save_path + 'itemcf_i2i_sim.pkl', 'wb'))
    return i2i_sim_

In [11]:
i2i_sim = itemcf_sim(all_click_df, item_created_time_dict)

100%|██████████| 10000/10000 [00:02<00:00, 4507.29it/s]


In [12]:
print(list(i2i_sim.keys())[0])

42567


简单的关联规则，如用户活跃度权重，这里将用户的点击次数作为用户活跃度的指标

In [13]:
# 得到用户活跃度
def get_user_activate_degree_dict(all_click_df):
    all_click_df_ = all_click_df.groupby('user_id')['click_article_id'].count().reset_index()
    
    # 用户活跃度归一化
    mm = MinMaxScaler()
    all_click_df_['click_article_id'] = mm.fit_transform(all_click_df_[['click_article_id']])
    user_activate_degree_dict = dict(zip(all_click_df_['user_id'], all_click_df_['click_article_id']))
    
    return user_activate_degree_dict

# usercf
def usercf_sim(all_click_df, user_activate_degree_dict):
    item_user_time_dict = get_item_user_time_dict(all_click_df)
    # 得到{item1: [(user1, t1), (user2, t2)],item2: [(user2, t3), (user3, t4)]}
    u2u_sim = {}  # u2u_sim[u][v]：用户u和v的相似度
    user_cnt = defaultdict(int) # user_cnt[u]：用户u点过多少文章
    for item, user_time_list in tqdm(item_user_time_dict.items()): #遍历文章item
        # 双循环，枚举点过同一篇文章的 (用户u, 用户v)
        for u, click_time  in user_time_list:
            user_cnt[u] += 1
            u2u_sim.setdefault(u, {})
            for v, click_time  in user_time_list:
                u2u_sim[u].setdefault(v, 0)
                if u == v:
                    continue
                # 相似度累加 用户平均活跃度作为活跃度的权重
                activate_weight = 100 * 0.5 * (user_activate_degree_dict[u] + user_activate_degree_dict[v])
                u2u_sim[u][v] += activate_weight / math.log(len(user_time_list) + 1)  # 分母：惩罚热门文章
    u2u_sim_ = u2u_sim.copy()   # {u1: {u2: 10, u3: 5}, u2: {u1: 10, u3: 2}, ...}   数字为加权共现次数
    for u, related_users in u2u_sim.items():
        for v, wij in related_users.items():
            # 归一化
            u2u_sim_[u][v] = wij / math.sqrt(user_cnt[u] * user_cnt[v])
            # 分母 惩罚过度活跃用户 相似度 = 共现次数 / sqrt(用户u活跃度 × 用户v活跃度)
    pickle.dump(u2u_sim_, open(save_path + 'usercf_u2u_sim.pkl', 'wb'))
    return u2u_sim_


user_activate_degree_dict = get_user_activate_degree_dict(all_click_df)
u2u_sim = usercf_sim(all_click_df, user_activate_degree_dict)

100%|██████████| 6628/6628 [00:01<00:00, 3354.48it/s] 


In [14]:
print(len(user_activate_degree_dict))
print(all_click_df['user_id'].nunique())

10000
10000


## item embedding sim   解决新文章冷启动问题
faiss搜索

In [15]:
# 向量检索相似度计算 
# faiss加速计算两两文章间的相似度
# topk指的是每个item, faiss搜索后返回最相似的topk个item
def embdding_sim(click_df, item_emb_df, save_path, topk):
    """
        输入文章 Embedding → 输出【内容相似矩阵】
        :param click_df: 数据表
        :param item_emb_df: 文章的embedding
        :param save_path: 保存路径
        :patam topk: 找最相似的topk篇
        return 文章相似性矩阵
        
        思路: 对于每一篇文章， 基于embedding的相似性返回topk个与其最相似的文章， 只不过由于文章数量太多，这里用了faiss进行加速
    """
    
    # 文章索引与文章id的字典映射，faiss只认数字索引
    item_idx_2_rawid_dict = dict(zip(item_emb_df.index, item_emb_df['article_id']))
    # 取出所有文章的 Embedding 向量，并转换为float32
    item_emb_cols = [x for x in item_emb_df.columns if 'emb' in x]
    item_emb_np = np.ascontiguousarray(item_emb_df[item_emb_cols].values, dtype=np.float32)
    # 向量单位化
    item_emb_np = item_emb_np / np.linalg.norm(item_emb_np, axis=1, keepdims=True)
    # 建立faiss索引
    item_index = faiss.IndexFlatIP(item_emb_np.shape[1])  # 余弦相似度，内积检索
    item_index.add(item_emb_np)

    # 批量检索每篇文章的 Top-K 相似文章
    # 相似度查询，给每个索引位置上的向量返回topk个item以及相似度
    sim, idx = item_index.search(item_emb_np, topk) # 返回列表
    
    # 组装成文章相似度字典  
    #{文章1: {文章2:0.9, 文章3:0.8...},
    # 文章2: {文章1:0.9, 文章5:0.7...}
    #                                }
    # 将向量检索的结果保存成原始id的对应关系
    item_sim_dict = collections.defaultdict(dict)
    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(range(len(item_emb_np)), sim, idx)):
        target_raw_id = item_idx_2_rawid_dict[target_idx]
        # 从1开始是为了去掉商品本身, 所以最终获得的相似商品只有topk-1
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]):    # [1:]跳过自己
            rele_raw_id = item_idx_2_rawid_dict[rele_idx]
            item_sim_dict[target_raw_id][rele_raw_id] = item_sim_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value
    
    # 保存i2i相似度矩阵
    pickle.dump(item_sim_dict, open(save_path + 'emb_i2i_sim.pkl', 'wb'))   
    
    return item_sim_dict

In [16]:
item_emb_df = pd.read_csv(data_path + '/articles_emb.csv')
emb_i2i_sim = embdding_sim(all_click_df, item_emb_df, save_path, topk=10) # topk可以自行设置

364047it [00:03, 97890.17it/s]


## Youtube DNN

In [17]:
# 负采样 获取训练数据
def gen_data_set(data, negsample = 0):
    data.sort_values('click_timestamp', inplace = True)
    item_ids = data['click_article_id'].unique()
    train_set = []
    test_set = []
    for reviewerID, hist in tqdm(data.groupby('user_id')): #挨个用户处理
        pos_list = hist['click_article_id'].tolist()  # 点过的所有文章转成列表
        if negsample > 0:
            # 生成负样本候选集
            candidate_set = list(set(item_ids) - set(pos_list))
            # 随机选负样本 负样本数量 = 正样本数量 * negsample
            neg_list = np.random.choice(candidate_set, size=len(pos_list)*negsample, replace=True)
        # 特殊情况：用户只点击了一篇文章
        if len(pos_list) == 1:
            # 此时必须放入训练集，否则用户/文章embedding无法学习到
            # 样本格式：(用户ID, 历史点击序列, 目标文章, 标签1=正, 历史长度)
            train_set.append(reviewerID, [pos_list[0]], pos_list[0], 1, len(pos_list))
            test_set.append(reviewerID, [pos_list[0]], pos_list[0], 1, len(pos_list))
        # 滑窗构造正负样本 用前 i 篇文章当历史，预测第 i+1 篇文章
        for i in range(1, len(pos_list)):
            hist = pos_list[:i] #当前历史为第i篇文章

            if i != len(pos_list)-1:    # i不是列表最后一个，放进训练集；否则放进测试集
                # 加入正样本 (用户ID, 倒序历史, 目标文章, 标签1, 历史长度)
                # hist[::-1]为历史倒序 让最近点击的文章排在最前面 [A, B, C] 变为[C, B, A]
                train_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))

                # 加负样本
                for negi in range(negsample):
                    train_set.append((reviewerID, hist[::-1], neg_list[i*negsample+negi], 0, len(hist[::-1])))
            else:
                # i是列表最后一个，只用完整最长历史做测试
                test_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set, test_set

# 将输入数据padding， 保证序列特征长度一致
# seq_max_len：历史序列最大长度
def gen_model_input(train_set,user_profile,seq_max_len):

    train_uid = np.array([line[0] for line in train_set])
    train_seq = [line[1] for line in train_set]
    train_iid = np.array([line[2] for line in train_set])
    train_label = np.array([line[3] for line in train_set])
    train_hist_len = np.array([line[4] for line in train_set])
    # 序列补齐  不够补0，超过截断
    train_seq_pad = pad_sequences(train_seq, maxlen=seq_max_len, padding='post', truncating='post', value=0)
    train_model_input = {"user_id": train_uid, "click_article_id": train_iid, "hist_article_id": train_seq_pad,
                         "hist_len": train_hist_len}

    return train_model_input, train_label


用 YouTubeDNN 模型训练用户和文章的向量（Embedding），然后用向量相似度给每个用户推荐最相似的 20 篇文章，最后保存结果、评估效果    

流程:    
把用户 ID、文章 ID 编码成数字    
生成训练样本（滑窗、历史序列→预测下一篇）    
搭建 YouTubeDNN 模型训练    
拿到 用户向量 和 文章向量    
用向量相似度搜索（faiss）给每个用户找最相似的文章    
保存向量 + 保存推荐结果    
评估召回效果

In [36]:
# 【最关键第一行：强制清空TF计算图，解决变量不存在报错】
tf.keras.backend.clear_session()
from tensorflow.keras.models import Model

# ===================== 你之前的两个工具函数（不动） =====================
def gen_data_set(data, negsample=0):
    data.sort_values("click_timestamp", inplace=True)
    item_ids = data['click_article_id'].unique()
    train_set = []
    test_set = []
    for reviewerID, hist in tqdm(data.groupby('user_id')):
        pos_list = hist['click_article_id'].tolist()
        if negsample > 0:
            candidate_set = list(set(item_ids) - set(pos_list))
            neg_list = np.random.choice(candidate_set,size=len(pos_list)*negsample,replace=True)
        if len(pos_list) == 1:
            train_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
            test_set.append((reviewerID, [pos_list[0]], pos_list[0],1,len(pos_list)))
        for i in range(1, len(pos_list)):
            hist = pos_list[:i]
            if i != len(pos_list) - 1:
                train_set.append((reviewerID, hist[::-1], pos_list[i], 1, len(hist[::-1])))
                for negi in range(negsample):
                    train_set.append((reviewerID, hist[::-1], neg_list[i*negsample+negi], 0,len(hist[::-1])))
            else:
                test_set.append((reviewerID, hist[::-1], pos_list[i],1,len(hist[::-1])))
    random.shuffle(train_set)
    random.shuffle(test_set)
    return train_set, test_set

def gen_model_input(train_set,user_profile,seq_max_len):
    train_uid = np.array([line[0] for line in train_set])
    train_seq = [line[1] for line in train_set]
    train_iid = np.array([line[2] for line in train_set])
    train_label = np.array([line[3] for line in train_set])
    train_hist_len = np.array([line[4] for line in train_set])
    train_seq_pad = pad_sequences(train_seq, maxlen=seq_max_len, padding='post', truncating='post', value=0)
    train_model_input = {"user_id": train_uid, "click_article_id": train_iid, "hist_article_id": train_seq_pad, "hist_len": train_hist_len}
    return train_model_input, train_label

# ===================== ✅ 100%修复 TF2.x YouTubeDNN =====================
def simple_youtube_dnn(user_vocab, item_vocab, seq_len, emb_dim=16):
    # 输入层（严格TF2规范）
    user_id = Input(shape=(1,), name="user_id", dtype="int32")
    item_id = Input(shape=(1,), name="click_article_id", dtype="int32")
    hist_item = Input(shape=(seq_len,), name="hist_article_id", dtype="int32")
    hist_len = Input(shape=(1,), name="hist_len", dtype="int32")

    # 用户Embedding
    user_emb = Embedding(user_vocab, emb_dim, name="user_emb")(user_id)
    user_emb_flat = Flatten()(user_emb)

    # 物品Embedding（共享）
    item_emb_layer = Embedding(item_vocab, emb_dim, name="item_emb")
    item_emb = item_emb_layer(item_id)
    item_emb_flat = Flatten()(item_emb)

    # 历史序列均值池化
    hist_emb = item_emb_layer(hist_item)
    hist_mean = GlobalAveragePooling1D()(hist_emb)

    # 拼接
    user_feature = Concatenate(axis=-1)([user_emb_flat, hist_mean])

    # DNN 层（命名空间绝对安全）
    dnn1 = Dense(64, activation="relu", name="dnn_1")(user_feature)
    user_out = Dense(emb_dim, name="user_out")(dnn1)

    # 内积得分
    dot = Multiply()([user_out, item_emb_flat])
    score = Lambda(lambda x: tf.reduce_sum(x, axis=1, keepdims=True))(dot)

    # 构建模型
    model = Model(
        inputs=[user_id, item_id, hist_item, hist_len],
        outputs=score
    )
    model.compile(optimizer="adam", loss="mse")

    # 推理模型
    user_model = Model(
        inputs=[user_id, hist_item, hist_len],
        outputs=user_out
    )
    item_model = Model(inputs=item_id, outputs=item_emb_flat)

    return model, user_model, item_model

# ===================== ✅ 主函数（修复所有冲突） =====================
def youtubednn_u2i_dict(data, topk=20):
    data = data.copy()
    # 每次进入函数强制清图，彻底解决变量不存在报错
    tf.keras.backend.clear_session()
    
    SEQ_LEN = 30
    user_profile_ = data[["user_id"]].drop_duplicates('user_id')
    item_profile_ = data[["click_article_id"]].drop_duplicates('click_article_id')  

    # ID编码
    features = ["click_article_id", "user_id"]
    feature_max_idx = {}
    for feature in features:
        lbe = LabelEncoder()
        data[feature] = lbe.fit_transform(data[feature])
        feature_max_idx[feature] = data[feature].max() + 1

    user_profile = data[["user_id"]].drop_duplicates('user_id')
    item_profile = data[["click_article_id"]].drop_duplicates('click_article_id')  
    user_index_2_rawid = dict(zip(user_profile['user_id'], user_profile_['user_id']))
    item_index_2_rawid = dict(zip(item_profile['click_article_id'], item_profile_['click_article_id']))

    # 构造数据
    train_set, test_set = gen_data_set(data, negsample = 5)
    train_model_input, train_label = gen_model_input(train_set, user_profile, SEQ_LEN)
    test_model_input, test_label = gen_model_input(test_set, user_profile, SEQ_LEN)

    embedding_dim = 16

    # 创建模型
    model, user_embedding_model, item_embedding_model = simple_youtube_dnn(
        user_vocab=feature_max_idx['user_id'],
        item_vocab=feature_max_idx['click_article_id'],
        seq_len=SEQ_LEN,
        emb_dim=embedding_dim
    )

    # 训练（TF2 原生，无初始化问题）
    model.fit(
        train_model_input, train_label,
        batch_size=512,
        epochs=5,
        verbose=1
    )

    # 生成Embedding
    all_item_model_input = {"click_article_id": item_profile['click_article_id'].values}
    user_embs = user_embedding_model.predict(test_model_input, batch_size=4096, verbose=1)
    item_embs = item_embedding_model.predict(all_item_model_input, batch_size=4096, verbose=1)

    # 归一化
    user_embs = user_embs / (np.linalg.norm(user_embs, axis=1, keepdims=True) + 1e-9)
    item_embs = item_embs / (np.linalg.norm(item_embs, axis=1, keepdims=True) + 1e-9)

    # 保存
    save_path = './'
    raw_user_id_emb_dict = {user_index_2_rawid[k]:v for k,v in zip(user_profile['user_id'], user_embs)}
    raw_item_id_emb_dict = {item_index_2_rawid[k]:v for k,v in zip(item_profile['click_article_id'], item_embs)}
    pickle.dump(raw_user_id_emb_dict, open(save_path+'user_youtube_emb.pkl','wb'))
    pickle.dump(raw_item_id_emb_dict, open(save_path+'item_youtube_emb.pkl','wb'))

    # FAISS
    index = faiss.IndexFlatIP(embedding_dim)
    index.add(item_embs)
    sim, idx = index.search(np.ascontiguousarray(user_embs), topk)

    user_recall_items_dict = collections.defaultdict(dict)
    for target_idx, sim_list, idx_list in tqdm(zip(test_model_input['user_id'], sim, idx)):
        uid = user_index_2_rawid[target_idx]
        for iid_idx, score in zip(idx_list, sim_list):
            iid = item_index_2_rawid[iid_idx]
            user_recall_items_dict[uid][iid] = score
    user_recall_items_dict = {k:sorted(v.items(),key=lambda x:x[1],reverse=True) for k,v in user_recall_items_dict.items()}
    
    pickle.dump(user_recall_items_dict, open(save_path+'youtube_u2i_dict.pkl','wb'))
    return user_recall_items_dict

# ===================== 调用 =====================
user_multi_recall_dict = {}
metric_recall = False
topk=20

# 运行！
user_multi_recall_dict['youtubednn_recall'] = youtubednn_u2i_dict(all_click_df, topk=20)
print("运行完成！推荐结果已保存")

100%|██████████| 10000/10000 [00:05<00:00, 1889.71it/s]


Epoch 1/5
419/419 [==============================] - 1s 1ms/step - loss: 0.0970
Epoch 2/5
419/419 [==============================] - 1s 1ms/step - loss: 0.0799
Epoch 3/5
419/419 [==============================] - 1s 1ms/step - loss: 0.0758
Epoch 4/5
419/419 [==============================] - 1s 1ms/step - loss: 0.0677
Epoch 5/5
2/2 [==============================] - 0s 2ms/step


10000it [00:00, 256287.82it/s]


运行完成！推荐结果已保存


## ItemCF recall

双路径 ItemCF 物品协同过滤召回，核心是 **「用户历史点击过的文章 → 找相似文章 → 加权打分 → 推荐」**：     
准备两种物品相似度矩阵：基于用户行为的传统 ItemCF 相似度、基于文章 Embedding 的内容相似度；          
对每个用户，遍历他的历史点击文章，取出每篇文章的 Top-K 相似文章；              
设计多权重融合策略：时间权重（新文章优先）+ 点击位置权重（最近点击权重更高）+ 内容相似度权重 + 基础协同相似度；           
对候选文章加权求和打分，过滤已点击文章，不足用热门文章补全，最终输出每个用户的 Top-N 召回列表；          
分别用行为相似度和Embedding 相似度跑了两套召回，既保证协同过滤的准确性，又保证内容相关性。

In [37]:
# 基于商品的召回i2i
def item_based_recommend(user_id, user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim):
    """
        基于文章协同过滤的召回
        :param user_id: 用户id
        :param user_item_time_dict: 字典, 根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
        :param i2i_sim: 字典，文章相似性矩阵
        :param sim_item_topk: 整数， 选择与当前文章最相似的前k篇文章
        :param recall_item_num: 整数， 最后的召回文章数量
        :param item_topk_click: 列表，点击次数最多的文章列表，用户召回补全
        :param emb_i2i_sim: 字典基于内容embedding算的文章相似矩阵
        
        return: 召回的文章列表 {item1:score1, item2: score2...}
        
    """
    # 获取用户历史交互的文章
    user_hist_items = user_item_time_dict[user_id]
    # 候选文章总得分
    item_rank = {}

    # 遍历用户的每一篇历史点击文章 loc为文章在历史序列中的位置，越新loc越大
    for loc, (i, click_time) in enumerate(user_hist_items):
        if i not in i2i_sim:  # <-- 加这一行！
            continue

        
        # 取出与文章i最相似的Top-K文章 j
        for j, wij in sorted(i2i_sim[i].items(), key=lambda x: x[1], reverse=True)[:sim_item_topk]:
            # 过滤 若相似文章j如果用户已经点过，则跳过
            if j in user_hist_items:
                continue
            
            # 四大权重计算
            # 1.文章创建时间差权重         指数衰减，两篇文章发布时间越近，权重越高  
            created_time_weight = np.exp(0.8 ** np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
            # 相似文章和历史点击文章序列中历史文章所在的位置权重          
            # 2.点击位置权重 → 最近点击的文章，权重更高（0.9的幂次衰减）
            loc_weight = (0.9 ** (len(user_hist_items) - loc))
            # 3.内容相似度权重 → 叠加Embedding向量相似度，增强内容相关性
            content_weight = 1.0
            if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                content_weight += emb_i2i_sim[i][j]
            if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                content_weight += emb_i2i_sim[j][i]
            # 4.基础相似度 wij（ItemCF/Embedding的原始分数）
            # 总得分 = 所有权重相乘，累加到候选文章j上
            item_rank.setdefault(j, 0)
            item_rank[j] += created_time_weight * loc_weight * content_weight * wij
    
    # 召回数量不足10个，用热门商品补全
    if len(item_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in item_rank.items(): # 填充的item应该不在原来的列表中
                continue
            # 赋负分以保证热门物品排在自定义召回后面
            
            item_rank[item] = - i - 100 # 随便给个负数就行
            if len(item_rank) == recall_item_num:
                break
    # 按照得分降序排序，返回top-n文章召回列表
    item_rank = sorted(item_rank.items(), key=lambda x: x[1], reverse=True)[:recall_item_num]
        
    return item_rank

In [38]:
# 初始化结果字典
user_recall_items_dict = {}

# 获取用户的点击历史（长数字版本）
user_item_time_dict = get_user_item_time(trn_hist_click_df)
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

# 循环给 10000 个用户做推荐
for user in tqdm(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user] = item_based_recommend(
        user, user_item_time_dict, i2i_sim, 20, 10,
        item_topk_click, item_created_time_dict, emb_i2i_sim
    )

100%|██████████| 10000/10000 [00:00<00:00, 12726.49it/s]


In [39]:
metrics_recall(user_recall_items_dict, trn_last_click_df, topk=10)

✅ 开始评估召回效果...
✅ Top@10 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@20 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@30 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@40 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@50 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000


1. 你为什么用 ItemCF 做召回？
ItemCF 适合新闻 / 文章场景：物品更新快、用户兴趣稳定，基于相似物品推荐比用户 CF 更稳定。
2. 你的权重设计为什么这么做？
时间权重：新闻时效性强，新文章优先；
位置权重：用户最近点击的文章，更代表当前兴趣；
内容权重：融合 Embedding，解决行为协同的冷启动 / 泛化问题；
加权相乘：放大优质候选，过滤低质候选。
3. 行为 ItemCF 和 Embedding 召回的区别？
行为 ItemCF：靠用户点击，精准但慢、有冷启动；
Embedding 召回：靠文章内容，快、泛化强、覆盖新文章；
我做了两套召回，后续可以融合，兼顾精准和效率。
4. 为什么要补全热门文章？
解决用户历史点击少、无相似文章的冷启动问题，保证每个用户都有推荐结果。

In [40]:
# 纯 ItemCF 行为协同召回（慢，精准）

# 先进行itemcf召回, 为了召回评估，所以提取最后一次点击

# 如果要评估召回效果：把数据拆成「历史点击」+「最后一次点击」（用最后一次点击验证推荐准不准）
# 如果不评估：直接用全量点击数据
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df
# 初始化存储结果：key=用户ID，value=召回的文章列表
user_recall_items_dict = collections.defaultdict(dict)
# 生成用户-历史点击字典：{用户1: {文章A:点击时间, 文章B:点击时间}}    给后面的权重计算（时间、位置）提供数据
user_item_time_dict = get_user_item_time(trn_hist_click_df)

In [41]:
missing_cnt = 0
for user, items in user_item_time_dict.items():
    for i, _ in items:
        if i not in i2i_sim:
            missing_cnt += 1

print("缺失item数量:", missing_cnt)

缺失item数量: 0


In [42]:
def metrics_recall(user_recall_items_dict, trn_last_click_df, topk=10):
    print("✅ 开始评估召回效果...")  # <-- 必加，确保函数执行了
    
    last_click_item_dict = dict(zip(trn_last_click_df['user_id'], trn_last_click_df['click_article_id']))
    user_num = len(user_recall_items_dict)

    # 固定评估 top10, top20... 不管你传什么，都能输出！
    for k in [10, 20, 30, 40, 50]:
        hit_num = 0
        for user, item_list in user_recall_items_dict.items():
            # 取前K个召回文章
            tmp_recall_items = [x[0] for x in item_list[:k]]
            if last_click_item_dict.get(user, -1) in tmp_recall_items:
                hit_num += 1
        
        hit_rate = round(hit_num / user_num, 5) if user_num > 0 else 0
        print(f'✅ Top@{k} | 命中数: {hit_num} | 召回率: {hit_rate} | 用户数: {user_num}')

# 必须打开评估！
metric_recall = True

# 加载数据（全部取消注释！不要注释任何一个！）
i2i_sim = pickle.load(open(r"C:\Users\Administrator\Desktop\tianchi\temp_results\itemcf_i2i_sim.pkl", 'rb'))
emb_i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl', 'rb'))  # 必须打开！

sim_item_topk = 20
recall_item_num = 10
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

user_recall_items_dict = {}

for user in tqdm(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user] = item_based_recommend(
        user, user_item_time_dict, i2i_sim, sim_item_topk, recall_item_num,
        item_topk_click, item_created_time_dict, emb_i2i_sim
    )

# 评估（必须传 topk=10）
if metric_recall:
    print("✅ 进入评估阶段...")
    metrics_recall(user_recall_items_dict, trn_last_click_df, topk=10)

100%|██████████| 10000/10000 [00:06<00:00, 1609.18it/s]

✅ 进入评估阶段...
✅ 开始评估召回效果...
✅ Top@10 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@20 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@30 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@40 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@50 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000


In [43]:
# 加载基于用户点击行为计算的物品相似度（纯ItemCF）
i2i_sim = pickle.load(open(r"C:\Users\Administrator\Desktop\tianchi\temp_results\itemcf_i2i_sim.pkl", 'rb'))
# i2i_sim = pickle.load(open(save_path + 'itemcf_i2i_sim.pkl', 'rb'))
# 加载基于文章embedding计算的内容相似度（用于加权加分，提升效果）
emb_i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl', 'rb'))

sim_item_topk = 20    # 每篇文章取20个最相似的文章
recall_item_num = 10  # 每个用户最终召回10篇文章
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)  # 取50个热门文章（补全用）
# 遍历所有用户，批量生成召回结果
for user in tqdm(trn_hist_click_df['user_id'].unique()):
    # # 调用核心召回函数，生成当前用户的推荐列表
    user_recall_items_dict[user] = item_based_recommend(user, user_item_time_dict,
                                                        i2i_sim, sim_item_topk, recall_item_num,
                                                        item_topk_click, item_created_time_dict, emb_i2i_sim)

# 把这套召回结果存入总字典，方便后续多策略融合
user_multi_recall_dict['itemcf_sim_itemcf_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['itemcf_sim_itemcf_recall'], open(save_path + 'itemcf_recall_dict.pkl', 'wb'))

# 评估开关：如果开启评估，用「最后一次点击」计算召回率
if metric_recall:
    # 召回效果评估
    metrics_recall(user_multi_recall_dict['itemcf_sim_itemcf_recall'], trn_last_click_df, topk=recall_item_num)

100%|██████████| 10000/10000 [00:06<00:00, 1633.46it/s]


✅ 开始评估召回效果...
✅ Top@10 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@20 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@30 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@40 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000
✅ Top@50 | 命中数: 4871 | 召回率: 0.4871 | 用户数: 10000


## embedding sim recall

In [44]:
# 这里是为了召回评估，所以提取最后一次点击
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)
i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl','rb'))

sim_item_topk = 20
recall_item_num = 10

item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

for user in tqdm(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user] = item_based_recommend(user, user_item_time_dict, i2i_sim, sim_item_topk, 
                                                        recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)
    
user_multi_recall_dict['embedding_sim_item_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['embedding_sim_item_recall'], open(save_path + 'embedding_sim_item_recall.pkl', 'wb'))

if metric_recall:
    # 召回效果评估
    metrics_recall(user_multi_recall_dict['embedding_sim_item_recall'], trn_last_click_df, topk=recall_item_num)

100%|██████████| 10000/10000 [00:00<00:00, 12612.71it/s]


✅ 开始评估召回效果...
✅ Top@10 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@20 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@30 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@40 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000
✅ Top@50 | 命中数: 171 | 召回率: 0.0171 | 用户数: 10000


## userCF recall

给用户推荐与其相似的用户历史点击文章    
可以加的关联规则：    
考虑相似用户的历史点击文章与被推荐用户历史点击商品的关系权重，而这里的关系就可以直接借鉴基于物品的协同过滤相似的做法，只不过这里是对被推荐物品关系的一个累加的过程，下面是使用的一些关系权重，及相关的代码：

计算被推荐用户历史点击文章与相似用户历史点击文章的相似度，文章创建时间差，相对位置的总和，作为各自的权重


第一段：user_based_recommend 函数           
→ UserCF 召回核心公式（给用户推荐 “相似用户喜欢的文章”）             
第二段：传统 UserCF 召回调用                
→ 基于用户点击行为算用户相似度（内存爆炸，几乎不用）           
第三段：Embedding UserCF 召回工具函数                   
→ 用 YoutubeDNN 用户向量 + FAISS 算用户相似度（工业界真正用的！）

In [45]:
# 基于用户的召回 u2u2i   给用户推荐 “和他兴趣相似的用户” 点击过的文章，并加权打分
def user_based_recommend(user_id, user_item_time_dict, u2u_sim, sim_user_topk, recall_item_num, 
                         item_topk_click, item_created_time_dict, emb_i2i_sim):
    """
        基于文章协同过滤的召回
        :param user_id: 要推荐的用户
        :param user_item_time_dict: 所有用户的历史点击字典
                                    根据点击时间获取用户的点击文章序列   {user1: {item1: time1, item2: time2..}...}
        :param u2u_sim: 字典，用户相似性矩阵
        :param sim_user_topk: 整数， 选择与当前用户最相似的前k个用户
        :param recall_item_num: 整数， 最后的召回文章数量
        :param item_topk_click: 列表，点击次数最多的文章列表，用户召回补全
        :param item_created_time_dict: 文章创建时间列表
        :param emb_i2i_sim: 字典基于内容embedding算的文章相似矩阵
        
        return: 召回的文章列表 {item1:score1, item2: score2...}
    """
    # 历史交互
    user_item_time_list = user_item_time_dict[user_id]    # {item1: time1, item2: time2...}
    user_hist_items = set([i for i, t in user_item_time_list])   # 一个用户与某篇文章的多次交互， 去重
    
    items_rank = {}
    if user_id not in u2u_sim:
        pass
    else:
        # 遍历【最相似的前 K 个用户】      sim_u = 相似用户         wuv = 相似度分数 
        for sim_u, wuv in sorted(u2u_sim[user_id].items(), key=lambda x: x[1], reverse=True)[:sim_user_topk]:
            if sim_u not in user_item_time_dict:
                continue
            for i, click_time in user_item_time_dict[sim_u]:
                if i in user_hist_items:
                    continue          # 过滤已读
                items_rank.setdefault(i, 0)

                # 给文章加权算分      总得分 = 所有权重 × 用户相似度
                loc_weight = 1.0                    # 位置权重（最近点击权重高）
                content_weight = 1.0                # 内容相似度权重
                created_time_weight = 1.0           # 时间权重（文章发布时间近权重高）
                
                # 当前文章与该用户看的历史文章进行一个权重交互
                for loc, (j, click_time) in enumerate(user_item_time_list):
                    # 点击时的相对位置权重
                    loc_weight += 0.9 ** (len(user_item_time_list) - loc)
                    # 内容相似性权重
                    if emb_i2i_sim.get(i, {}).get(j, None) is not None:
                        content_weight += emb_i2i_sim[i][j]
                    if emb_i2i_sim.get(j, {}).get(i, None) is not None:
                        content_weight += emb_i2i_sim[j][i]
                    
                    # 创建时间差权重
                    created_time_weight += np.exp(0.8 * np.abs(item_created_time_dict[i] - item_created_time_dict[j]))
                # 分数累加 → 排序 → 返回 TopN 
                items_rank[i] += loc_weight * content_weight * created_time_weight * wuv
        
    # 热度补全
    if len(items_rank) < recall_item_num:
        for i, item in enumerate(item_topk_click):
            if item in items_rank.items(): # 填充的item应该不在原来的列表中
                continue
            items_rank[item] = - i - 100 # 随便给个复数就行
            if len(items_rank) == recall_item_num:
                break
        
    items_rank = sorted(items_rank.items(), key=lambda x: x[1], reverse=True)[:recall_item_num]    
    
    return items_rank


### 传统 UserCF，基于用户点击重叠计算用户相似度。


In [46]:
# 这里是为了召回评估，所以提取最后一次点击
# 由于usercf中计算user之间的相似度的过程太费内存了，全量数据这里就没有跑，跑了一个采样之后的数据
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df
    
user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)

u2u_sim = pickle.load(open(save_path + 'usercf_u2u_sim.pkl', 'rb'))

sim_user_topk = 20
recall_item_num = 10
item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

for user in tqdm(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user] = user_based_recommend(user, user_item_time_dict, u2u_sim, sim_user_topk,
                                                        recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)    

pickle.dump(user_recall_items_dict, open(save_path + 'usercf_u2u2i_recall.pkl', 'wb'))

if metric_recall:
    # 召回效果评估
    metrics_recall(user_recall_items_dict, trn_last_click_df, topk=recall_item_num)

100%|██████████| 10000/10000 [00:35<00:00, 282.49it/s]

✅ 开始评估召回效果...
✅ Top@10 | 命中数: 6418 | 召回率: 0.6418 | 用户数: 10000
✅ Top@20 | 命中数: 6418 | 召回率: 0.6418 | 用户数: 10000
✅ Top@30 | 命中数: 6418 | 召回率: 0.6418 | 用户数: 10000
✅ Top@40 | 命中数: 6418 | 召回率: 0.6418 | 用户数: 10000
✅ Top@50 | 命中数: 6418 | 召回率: 0.6418 | 用户数: 10000


### user embedding sim召回
验证上述基于用户的协同过滤的代码，下面使用了YoutubeDNN过程中产生的user embedding来进行向量检索每个user最相似的topk个user，在使用这里得到的u2u的相似性矩阵，使用usercf进行召回     

使用 YoutubeDNN 训练出的用户 Embedding，     
通过 FAISS 快速检索相似用户，        
生成 用户相似度矩阵，       
解决传统 UserCF 计算慢、耗内存 的问题

In [47]:
# 使用Embedding的方式获取u2u的相似性矩阵
# topk指的是每个user, faiss搜索后返回最相似的topk个user
def u2u_embdding_sim(click_df, user_emb_dict, save_path, topk):
    '''
    输入：youtubeDNN输出的用户向量 user_emb_dict
    输出：用户相似度矩阵 youtube_u2u_sim.pkl
    '''
    user_list = []
    user_emb_list = []
    # 把用户 ID 和用户向量整理成列表 丢进 FAISS 以便给每个用户快速找到 Top-K 最相似的用户
    for user_id, user_emb in user_emb_dict.items():
        user_list.append(user_id)
        user_emb_list.append(user_emb)
        
    user_index_2_rawid_dict = {k: v for k, v in zip(range(len(user_list)), user_list)}    
    
    user_emb_np = np.array(user_emb_list, dtype=np.float32)
    
    # 建立faiss索引
    user_index = faiss.IndexFlatIP(user_emb_np.shape[1])
    user_index.add(user_emb_np)
    # 相似度查询，给每个索引位置上的向量返回topk个item以及相似度
    sim, idx = user_index.search(user_emb_np, topk) # 返回的是列表
   
    # 将向量检索的结果保存成原始id的对应关系
    user_sim_dict = collections.defaultdict(dict)
    for target_idx, sim_value_list, rele_idx_list in tqdm(zip(range(len(user_emb_np)), sim, idx)):
        target_raw_id = user_index_2_rawid_dict[target_idx]
        # 从1开始是为了去掉商品本身, 所以最终获得的相似商品只有topk-1
        for rele_idx, sim_value in zip(rele_idx_list[1:], sim_value_list[1:]): 
            rele_raw_id = user_index_2_rawid_dict[rele_idx]
            user_sim_dict[target_raw_id][rele_raw_id] = user_sim_dict.get(target_raw_id, {}).get(rele_raw_id, 0) + sim_value
    
    # 保存i2i相似度矩阵
    pickle.dump(user_sim_dict, open(save_path + 'youtube_u2u_sim.pkl', 'wb'))   
    return user_sim_dict

In [48]:
# 读取YoutubeDNN过程中产生的user embedding, 然后使用faiss计算用户之间的相似度
# 这里需要注意，这里得到的user embedding其实并不是很好，因为YoutubeDNN中使用的是用户点击序列来训练的user embedding,
# 如果序列普遍都比较短的话，其实效果并不是很好
user_emb_dict = pickle.load(open('user_youtube_emb.pkl', 'rb'))
u2u_sim = u2u_embdding_sim(all_click_df, user_emb_dict, save_path, topk=10)

10000it [00:00, 97496.61it/s]


In [51]:
# 使用召回评估函数验证当前召回方式的效果
if metric_recall:
    trn_hist_click_df, trn_last_click_df = get_hist_and_last_click(all_click_df)
else:
    trn_hist_click_df = all_click_df

user_recall_items_dict = collections.defaultdict(dict)
user_item_time_dict = get_user_item_time(trn_hist_click_df)
u2u_sim = pickle.load(open(save_path + 'youtube_u2u_sim.pkl', 'rb'))

sim_user_topk = 20
recall_item_num = 10

item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)
for user in tqdm(trn_hist_click_df['user_id'].unique()):
    user_recall_items_dict[user] = user_based_recommend(user, user_item_time_dict, u2u_sim, sim_user_topk, \
                                                        recall_item_num, item_topk_click, item_created_time_dict, emb_i2i_sim)
    
user_multi_recall_dict['youtubednn_usercf_recall'] = user_recall_items_dict
pickle.dump(user_multi_recall_dict['youtubednn_usercf_recall'], open(save_path + 'youtubednn_usercf_recall.pkl', 'wb'))

if metric_recall:
    # 召回效果评估
    metrics_recall(user_multi_recall_dict['youtubednn_usercf_recall'], trn_last_click_df, topk=recall_item_num)

100%|██████████| 10000/10000 [00:02<00:00, 4380.19it/s]

✅ 开始评估召回效果...
✅ Top@10 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@20 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@30 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@40 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@50 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000


下面两个单元都是调试用的

In [49]:
# 体检 1：看看字典里是不是真的存了 10,000 个用户
print(f"用户总数: {len(u2u_sim)}")

# 体检 2：随便看一个用户的推荐结果，看看有没有分数
sample_user = list(u2u_sim.keys())[10]
print(f"样本用户 {sample_user} 的相似用户: {u2u_sim[sample_user]}")

# 体检 3：看看相似度分数
# 如果分数值全都是 0 或者全都是同一个数，那才有问题

用户总数: 10000
样本用户 199783 的相似用户: {181443: 0.9713859558105469, 194889: 0.9687173962593079, 91040: 0.9633815288543701, 12203: 0.9614738821983337, 6663: 0.9611104130744934, 155216: 0.9592500925064087, 28696: 0.9573255181312561, 53494: 0.9566430449485779, 110217: 0.9565672874450684}


In [50]:
# 1. 初始化一个字典来存这一路召回的结果
user_recall_items_dict_usercf = {}

# 2. 设置参数（跟之前保持一致，或者稍微调大一点）
sim_user_topk = 10     # 每个用户找 10 个最相似的邻居
recall_item_num = 10   # 每个用户最终推荐 10 篇文章

# 3. 开始循环（如果你之前没跑过这几行，确保它们在内存里）
# user_item_time_dict = get_user_item_time(trn_hist_click_df)
# item_topk_click = get_item_topk_click(trn_hist_click_df, k=50)

print(" 开始 UserCF 召回...")
for user in tqdm(trn_hist_click_df['user_id'].unique()):
    # 注意这里调用的是 user_based_recommend
    user_recall_items_dict_usercf[user] = user_based_recommend(
        user, 
        user_item_time_dict, 
        u2u_sim,              # 这里就是你那个全是 0.99 的矩阵
        sim_user_topk, 
        recall_item_num, 
        item_topk_click, 
        item_created_time_dict, 
        emb_i2i_sim
    )
# 运行评估函数，看看这路召回准不准
metrics_recall(user_recall_items_dict_usercf, trn_last_click_df, topk=10)

 开始 UserCF 召回...


100%|██████████| 10000/10000 [00:02<00:00, 4498.01it/s]

✅ 开始评估召回效果...
✅ Top@10 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@20 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@30 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@40 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000
✅ Top@50 | 命中数: 205 | 召回率: 0.0205 | 用户数: 10000


## 冷启动

#### 冷启动问题大致分类：
文章冷启动：对于一个平台系统新加入的文章，该文章没有任何的交互记录，如何推荐给用户的问题。(对于我们场景可以认为是，日志数据中没有出现过的文章都可以认为是冷启动的文章)              
用户冷启动：对于一个平台系统新来的用户，该用户还没有文章的交互信息，如何给该用户进行推荐。(对于我们场景就是，测试集中的用户是否在测试集对应的log数据中出现过，如果没有出现过，那么可以认为该用户是冷启动用户。但是有时候并没有这么严格，我们也可以自己设定某些指标来判别哪些用户是冷启动用户，比如通过使用时长，点击率，留存率等等)              
系统冷启动：就是对于一个平台刚上线，还没有任何的相关历史数据，此时就是系统冷启动，其实也就是前面两种的一个综合           

数据集里总共有 30 万篇文章，但用户点击日志里只出现了 3 万多篇。      
也就是说：          
有 27 万篇文章从来没被点过 → 这就是物品冷启动。             
同时，测试集里 20% 的用户只有 1 次点击，行为极少 → 这是用户冷启动。           
如果你的召回只从 “有点击的 3 万篇” 里找，就永远推荐不到那 27 万新文章，也无法满足新用户。          
所以必须做：冷启动召回 = 给用户推荐从未出现在点击日志里的新文章。

#### 文章冷启动 思路：    
从27w无行为文章里，挑选用户可能喜欢的  
1. 用 文章 Embedding 找出和用户历史点击内容最相似的新文章      
2. 用规则过滤，只保留更符合用户偏好的：      
主题相同    
文章长度相近        
发布时间更近（新文章、热点文章）      
把这些文章作为冷启动召回候选，补充进推荐列表
      
意义：    
覆盖无行为新文章      
解决推荐生态 “只推老热门” 的问题      
增强内容多样性          

### 用户冷启动 思路：     
测试集中 20% 用户只有 1 次点击，行为太少，模型很难学。     
可行策略：        
对行为极少的用户，不依赖协同过滤，改用规则 / 内容召回      
用用户的唯一一次点击做内容相似推荐   
直接补充热点文章     
不进模型排序，用规则兜底         

### 和普通 Embedding 召回的区别（面试必问，超精简）            
普通 Embedding 召回：   
只从有点击日志的文章里找相似      
目的是提高精准度    
冷启动 Embedding 召回：   
从全部 30 万文章里找相似         
专门挑从未出现在点击日志里的新文章   
候选数量更多，再用规则过滤               
目的是覆盖冷启动文章，提升多样性

In [52]:
# cold start recall 补全当前协同过滤找回不到的无点击行为的新文章    
# 1.粗召回（用文章embedding相似度 给每个用户大量召回相似文章）
# 2.精过滤（规则兜底筛选）    用 4 条业务规则，只保留符合用户偏好、且从未出现在点击日志里的冷启动新文章，最终形成冷启动专属推荐列表
# 先多召、后精筛：避免规则太严格导致筛空，保证每个用户都有冷启动推荐结果

变量名	来源	作用               
item_type_dict	文章信息表	key = 文章 id，value = 文章分类 / 主题       
item_words_dict	文章信息表	key = 文章 id，value = 文章字数       
item_created_time_dict	文章信息表	key = 文章 id，value = 文章发布时间戳       
emb_i2i_sim.pkl	提前计算	基于文章内容 Embedding算的物品相似度

In [53]:
# 冷启动粗召回

# 用全量点击数据，生成冷启动候选、
trn_hist_click_df = all_click_df
# 空字典：存储每个用户的冷启动原始候选文章
user_recall_items_dict = collections.defaultdict(dict)
# 【用户-点击文章-时间】字典
user_item_time_dict = get_user_item_time(trn_hist_click_df)

# 内容Embedding相似度
i2i_sim = pickle.load(open(save_path + 'emb_i2i_sim.pkl','rb'))
# 超参数  冷启动必须多召回 数值设大点
sim_item_topk = 150         # 每篇历史文章，找150篇相似文章
recall_item_num = 100       # 每个用户先召回100篇候选

# 遍历所有用户，批量生成冷启动候选
for user in tqdm(trn_hist_click_df['user_id'].unique()):
    # 调用ItemCF召回函数，但传入的是【内容相似度】，不是行为相似度
    user_recall_items_dict[user] = item_based_recommend(
        user, 
        user_item_time_dict, 
        i2i_sim,          # 重点：内容Embedding相似度
        sim_item_topk, 
        recall_item_num, 
        item_topk_click,
        item_created_time_dict, 
        emb_i2i_sim
    )

# 保存原始候选（未经过滤，后面要用）
pickle.dump(user_recall_items_dict, open(save_path + 'cold_start_items_raw_dict.pkl', 'wb'))

100%|██████████| 10000/10000 [00:01<00:00, 8596.21it/s]


In [54]:
# 冷启动规则过滤

# 函数：返回一个集合，包含所有在点击日志里出现过的文章id
def get_click_article_ids_set(all_click_df):
    return set(all_click_df.click_article_id.values)

# 冷启动过滤函数
def cold_start_items(
    user_recall_items_dict,    # 上一步召回的100篇原始候选
    user_hist_item_typs_dict,  # 用户历史点击的文章类型
    user_hist_item_words_dict, # 用户历史点击的平均字数
    user_last_item_created_time_dict, # 用户最后一次点击的文章时间
    item_type_dict,    # 文章-类型字典
    item_words_dict,   # 文章-字数字典
    item_created_time_dict, # 文章-发布时间字典
    click_article_ids_set, # 所有旧文章集合
    recall_item_num   # 最终冷启动要保留多少篇
):
    # 新建空字典：存储【最终过滤完成的冷启动文章】
    cold_start_user_items_dict = {}

    # 遍历每一个用户 + 他的100篇候选文章·~1·
    for user, item_list in tqdm(user_recall_items_dict.items()):
        # 给当前用户初始化空列表
        cold_start_user_items_dict.setdefault(user, [])

        # 遍历候选文章（item=文章id，score=相似度分数）
        for item, score in item_list:
            # ===================== 提取【用户的历史偏好】 =====================
            hist_item_type_set = user_hist_item_typs_dict[user]          # 用户喜欢的文章类型
            hist_mean_words = user_hist_item_words_dict[user]             # 用户喜欢的平均字数
            hist_last_time = user_last_item_created_time_dict[user]      # 最后点击时间戳
            hist_last_time = datetime.fromtimestamp(hist_last_time)      # 转成标准时间

            # ===================== 提取【候选文章的信息】 =====================
            curr_item_type = item_type_dict[item]        # 当前候选文章类型
            curr_item_words = item_words_dict[item]      # 当前候选文章字数
            curr_item_time = item_created_time_dict[item]# 当前候选文章发布时间
            curr_item_time = datetime.fromtimestamp(curr_item_time) # 转标准时间

            # ===================== 四大规则筛选：不满足直接跳过 =====================
            if (
                # 规则1：文章类型 不在 用户喜欢的类型里 → 跳过
                curr_item_type not in hist_item_type_set 
                # 规则2：文章是【旧文章】（出现过点击）→ 跳过（核心！只留新文章）
                or item in click_article_ids_set
                # 规则3：文章字数 和 用户历史平均字数 差>200 → 跳过
                or abs(curr_item_words - hist_mean_words) > 200
                # 规则4：文章发布时间 距离最后点击超过90天 → 跳过（新闻要新）
                or abs((curr_item_time - hist_last_time).days) > 90
            ): 
                continue

            # ===================== 满足所有规则 → 加入冷启动列表 =====================
            cold_start_user_items_dict[user].append((item, score))

    # 最后一步：按相似度分数排序，只保留前N篇
    cold_start_user_items_dict = {
        k: sorted(v, key=lambda x:x[1], reverse=True)[:recall_item_num] 
        for k, v in cold_start_user_items_dict.items()
    }

    # 保存最终冷启动结果
    pickle.dump(cold_start_user_items_dict, open(save_path + 'cold_start_user_items_dict.pkl', 'wb'))
    
    return cold_start_user_items_dict

In [55]:
all_click_df_ = all_click_df.copy()
# 把点击日志 和 文章信息合并（拿到类型、字数、时间）
all_click_df_ = all_click_df_.merge(item_info_df, how='left', on='click_article_id')
# 生成4个用户偏好字典
user_hist_item_typs_dict, user_hist_item_ids_dict, user_hist_item_words_dict, user_last_item_created_time_dict = get_user_hist_item_info_dict(all_click_df_)
# 所有旧文章集合
click_article_ids_set = get_click_article_ids_set(all_click_df)
# 需要注意的是
# 这里使用了很多规则来筛选冷启动的文章，所以前面再召回的阶段就应该尽可能的多召回一些文章，否则很容易被删掉
cold_start_user_items_dict = cold_start_items(user_recall_items_dict, user_hist_item_typs_dict, user_hist_item_words_dict, \
                                              user_last_item_created_time_dict, item_type_dict, item_words_dict, \
                                              item_created_time_dict, click_article_ids_set, recall_item_num)

# 把冷启动召回，加入多路召回集合（作为第4路召回）
user_multi_recall_dict['cold_start_recall'] = cold_start_user_items_dict

100%|██████████| 10000/10000 [00:00<00:00, 19606.65it/s]


## 多路召回合并    
将前面所有的召回策略得到的用户文章列表合并起来，下面是对前面所有召回结果的汇总

基于itemcf计算的item之间的相似度sim进行的召回        
基于embedding搜索得到的item之间的相似度进行的召回                  
YoutubeDNN召回          
YoutubeDNN得到的user之间的相似度进行的召回        
基于冷启动策略的召回

In [56]:
# 函数：合并多路召回结果 → 加权融合 → 排序 → 输出最终候选集

# user_multi_recall_dict    5路召回的总结果字典
# weight_dict  人工定义的每路召回权重
# topk   最终每个用户保留多少篇文章
def combine_recall_results(user_multi_recall_dict, weight_dict=None, topk=25):
    # 【新建变量】最终融合后的总字典（空）
    final_recall_items_dict = {}

    # 工具函数：分数归一化 
    # 作用：把每一路召回的分数，统一缩放到 0~1 之间，保证不同路召回公平融合
    # 输入：单用户的排序后文章列表 [(文章id, 原始分数), ...]
    def norm_user_recall_items_sim(sorted_item_list):
        # 判断1：如果文章数量<2，无法归一化，直接返回原数据
        # 原因：冷启动可能筛完没文章，避免计算报错
        if len(sorted_item_list) < 2:
            return sorted_item_list
        
        # 【新建变量】获取当前用户这路召回的 最小分数、最大分数
        min_sim = sorted_item_list[-1][1]  # 列表最后一个是分数最低的
        max_sim = sorted_item_list[0][1]   # 列表第一个是分数最高的
        
        # 【新建变量】存储归一化后的结果列表
        norm_sorted_item_list = []
        
        # 遍历每一篇文章，逐一分归一化
        for item, score in sorted_item_list:
            if max_sim > 0:
                # 归一化公式：(原始分数-最小分)/(最大分-最小分) → 缩放到0~1
                # 如果最大分=最小分（所有分数一样），直接赋值1.0
                norm_score = 1.0 * (score - min_sim) / (max_sim - min_sim) if max_sim > min_sim else 1.0
            else:
                # 最大分≤0，无有效分数，赋值0
                norm_score = 0.0
            
            # 把归一化后的(文章id, 新分数)加入列表
            norm_sorted_item_list.append((item, norm_score))
        
        # 返回归一化后的列表
        return norm_sorted_item_list

    # 遍历每一路召回，开始融合
    print('多路召回合并...')
    # 遍历 总召回字典：method=召回通道名称，user_recall_items=该通道的所有用户结果
    for method, user_recall_items in tqdm(user_multi_recall_dict.items()):
        print(method + '...')
        
        # ===================== 权重分配（人为定义！） =====================
        # 如果没传权重字典，所有通道权重=1（一视同仁）
        if weight_dict == None:
            recall_method_weight = 1
        # 如果传了，取对应通道的人工权重
        else:
            recall_method_weight = weight_dict[method]
        
        # ===================== 第一步：对当前路的所有用户做分数归一化 =====================
        # 遍历该路召回的所有用户，替换成归一化后的分数
        for user_id, sorted_item_list in user_recall_items.items():
            user_recall_items[user_id] = norm_user_recall_items_sim(sorted_item_list)
        
        # ===================== 第二步：加权分数累加融合 =====================
        # 遍历该路召回的所有用户
        for user_id, sorted_item_list in user_recall_items.items():
            # 给最终字典初始化当前用户（没有就创建空字典）
            final_recall_items_dict.setdefault(user_id, {})
            
            # 遍历该用户的所有文章+归一化分数
            for item, score in sorted_item_list:
                # 给当前文章初始化分数（没有就设为0）
                final_recall_items_dict[user_id].setdefault(item, 0)
                # 核心：最终分数 = 原始分数 × 该路召回的权重 → 累加到总分数
                final_recall_items_dict[user_id][item] += recall_method_weight * score  

    # ===================== 第三步：排序 + 截取TopK =====================
    # 【新建变量】存储最终排序后的结果
    final_recall_items_dict_rank = {}
    
    # 遍历所有用户，对文章按总分降序排序，只保留前topk篇
    for user, recall_item_dict in final_recall_items_dict.items():
        final_recall_items_dict_rank[user] = sorted(
            recall_item_dict.items(),  # 文章+总分数
            key=lambda x: x[1],       # 按分数排序
            reverse=True              # 降序（高分在前）
        )[:topk]  # 只保留前topk篇（150篇）

    # ===================== 第四步：保存最终结果 =====================
    pickle.dump(final_recall_items_dict, open(os.path.join(save_path, 'final_recall_items_dict.pkl'),'wb'))

    # 返回最终排序后的召回结果
    return final_recall_items_dict_rank

# ===================== 调用函数：执行多路召回合并 =====================
# 【人工定义权重！】5路召回权重全部设为1.0（一视同仁）
weight_dict = {'itemcf_sim_itemcf_recall': 1.0,
               'embedding_sim_item_recall': 1.0,
               'youtubednn_recall': 1.0,
               'youtubednn_usercf_recall': 1.0, 
               'cold_start_recall': 1.0}

# 调用函数：合并所有召回，每个用户最终保留150篇文章
final_recall_items_dict_rank = combine_recall_results(user_multi_recall_dict, weight_dict, topk=150)

多路召回合并...


  0%|          | 0/5 [00:00<?, ?it/s]

youtubednn_recall...


 60%|██████    | 3/5 [00:00<00:00,  6.55it/s]

itemcf_sim_itemcf_recall...
embedding_sim_item_recall...
youtubednn_usercf_recall...
cold_start_recall...


100%|██████████| 5/5 [00:00<00:00,  7.04it/s]


做归一化的原因：      
每路召回的分数范围不一样， 之间相加不公平，归一化把所有分数缩放到0-1可以保证每路召回平等参与加权     

最终总分数 = 路1分数×权重1 + 路2分数×权重2 + ... + 路5分数×权重5          

final_recall_items_dict_rank       
{        
  用户1: [(文章A, 总分9.5), (文章B, 总分8.2), ... 共150篇],      
  用户2: [(文章C, 总分9.0), (文章D, 总分7.8), ... 共150篇]                
}